# ORAX-KG — Pipeline Notebook

End-to-end pipeline.

**Stages:**
1. Ontology-guided triple extraction (LLM)
2. Triple embedding & alignment
3. Consensus clustering
4. LLM cluster validation → Ontology expansion with novel relations


---
## Step 1 — Ontology-Guided Triple Extraction

Loads the relation schema, splits it into known/hidden sets, then runs the LLM on each sentence to extract `(subject, relation, object)` triples.

**Output:** `results/notebook_run/02_extraction/extractions.jsonl

In [21]:
! python scripts/run_extraction.py --data data/raw_data/DATASET_EXAMPLE.json --schema data/ontologies/ReTACRED_ontology.json --output results/test_run --base-url https://unflippant-concetta-bilgiest.ngrok-free.dev/v1 --model Qwen/Qwen3-14B



  Run started: 2026-03-12 14:12:21
  Log: results/logs/test_run_log\extraction.log




ORAX-KG Relation Extraction:

  Output directory: results\test_run

  Connecting to vLLM server at https://unflippant-concetta-bilgiest.ngrok-free.dev/v1...
  Model: Qwen/Qwen3-14B

  Loading data from data/raw_data/DATASET_EXAMPLE.json...
   Loaded 296 triples

  Loading schema from data/ontologies/ReTACRED_ontology.json...
   Loaded schema from data/ontologies/ReTACRED_ontology.json
   Domain types: 2
   Total relation types: 120
   Loaded 95 ontology entries
   Found 23 relations with multiple type signatures

  Splitting ontology...

 Ontology Split:
   Total ontology entries: 95
   Known:  31 relations -> 71 entries
   Hidden: 8 relations -> 24 entries

  Saving ontology splits...
   Saved to results\test_run\01_ontology

  Sampling test data...
Sampling Statistics:
   Known samples:  167
   Hidden samples: 129
   Total sampled:  296

  Starting fresh extraction

  Extracting from index 0...
 

---
## Step 2 — Triple Embedding & Alignment

Embeds extracted triples and known ontology patterns into four independent views, then aligns each triple against the ontology using cosine similarity.

**Output:** `results/notebook_run/03_alignment/`

In [22]:
! python scripts/run_alignment.py --run-dir results/test_run --output results/test_run --embedder sentence-transformers/all-MiniLM-L6-v2 --threshold 0.90 --device cpu



  Run started: 2026-03-12 14:23:01
  Log: results/logs/test_run_log\alignment.log




ORAX-KG Ontology Alignment:

  Loading from:     results\test_run
  Output directory: results\test_run\03_alignment

  Loading extraction artifacts...
   Loaded 296 extractions
   Loaded known ontology (71 entries)

  Preparing extracted triples...

  Initializing embedder: sentence-transformers/all-MiniLM-L6-v2

  Embedding extracted triples...

  Embedding ontology patterns...

  Saving embeddings...
   Saved to results\test_run\03_alignment

  Computing similarities...

  Aligning triples (threshold=0.9)...

  Computing alignment metrics...


ALIGNMENT EVALUATION REPORT:

Precision:    0.927
Recall:       0.838
F1-Score:     0.881
Specificity:  0.915

True Positives:  140
False Positives: 11
False Negatives: 27
True Negatives:  118
Total:           296


  Alignment complete!
   Aligned:          151
   Novel candidates: 145
   Artifacts saved:  results\test_run\03_alignment


`torch_dtype` is deprecated! Use `dtype` instead!


---
## Step 3 — Consensus Clustering

Groups unaligned triples into semantically coherent clusters using a multi-algorithm ensemble (Spectral, HDBSCAN, Leiden).

**Output:** `results/notebook_run/04_clustering/`

In [23]:
! python scripts/run_clustering.py --run-dir results/test_run --similarity-threshold 0.60 --consensus-runs 3



  Run started: 2026-03-12 14:25:59
  Log: results/logs/test_run_log\clustering.log




ORAX-KG Consensus Clustering:

  Loading from: results\test_run

  Loading artifacts...
Loaded 296 extractions, 296 alignment results, 296 embeddings

  Computing inter-triple similarities...

  Running consensus clustering...
 Initialized with algorithms: ['spectral', 'hdbscan', 'leiden']
   Type stratification: 12 type-pair groups
   Clustering organization:person (29 items)
   Clustering person:person (22 items)
   Clustering person:title (14 items)
   Clustering person:number (13 items)
   Clustering person:nationality (19 items)
   Clustering person:criminal_charge (15 items)
   Clustering organization:city (2 items)
   Clustering person:country (5 items)
   Clustering person:organization (18 items)
   Clustering organization:country (2 items)
   Preserving 17 singletons as potential novel relations
→ Mode 'relation' produced 29 clusters
   Preserving 0 singletons as potential novel relations


---
## Step 4 — LLM Cluster Validation

For each cluster, the LLM applies a four-step protocol to decide whether it represents a genuinely novel ontology relation.

**Output:** `results/notebook_run/05_validation/`

In [26]:
! python scripts/run_validation.py --extraction-dir results/test_run --clusters results/test_run/04_clustering/clusters.json --embeddings results/test_run/03_alignment/extracted_embeddings.pt --output results/test_run/05_validation --base-url https://unflippant-concetta-bilgiest.ngrok-free.dev/v1 --model Qwen/Qwen3-14B



  Run started: 2026-03-12 14:47:19
  Log: results/logs/test_run_log\validation.log




ORAX-KG Cluster Validation:

  Extraction dir: results\test_run
  Output dir:     results\test_run\05_validation

  Connecting to vLLM server at https://unflippant-concetta-bilgiest.ngrok-free.dev/v1...
  Model: Qwen/Qwen3-14B

  Loading ontology...
   Known relations:  71
   Ontology classes: 16

  Loading clusters from results/test_run/04_clustering/clusters.json...
   Loaded clusters from results/test_run/04_clustering/clusters.json
   Relation-mode clusters: 29

  Loading embeddings from results/test_run/03_alignment/extracted_embeddings.pt...
   Loaded 296 embeddings from results/test_run/03_alignment/extracted_embeddings.pt

  Computing inter-triple similarities...

  Loading cluster embeddings from results\test_run\04_clustering\cluster_embeddings.pt...
   Loaded 141 items across 29 relation clusters


VALIDATING 29 CLUSTERS


Cluster 0: Skipping (size 1 < min 5)

Cluster 1 (26 items):
   Na

---
## Full Pipeline

Runs all four stages end-to-end from the config file.

In [ ]:
! python scripts/run_full_pipeline.py --config configs/test_config.yaml